# Assignment I: Trickle-Down vs Capital Gains

One section per question. Complete the `.py` files and the cells where it says `write your code here`. The tables and figures are given: please use the provided code to plot figures and tables.

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from HANCTobinModel import HANCTobinModelClass
import steady_state
import tails

## Settings

In [ ]:
plt.rcParams.update({'axes.grid':True,'grid.alpha':0.3,'figure.figsize':(6,4)})
os.makedirs('figs',exist_ok=True)

Z_1970 = steady_state.permanent_income_grid(1970)[0]
Z_2019 = steady_state.permanent_income_grid(2019)[0]
data_1970 = steady_state.wealth_shares_data(1970)

T_plot = 100 # years shown in figures

**Memory.** Every model holds about 0.35 GB of transition-path arrays (`path.a`, `path.D`, ...), and `model.copy()` and `copy.deepcopy` duplicate them. With a dozen models alive, the notebook needs more than 4 GB. Delete models you no longer need with `del`, and keep only the parts of a path you use later.

## 1. Calibration

### 1(a) Pareto tails

In [ ]:
model = HANCTobinModelClass(name='baseline')
par = model.par
ss = model.ss

par.xi = np.nan # write your code here
par.Sigma = np.nan # write your code here

print(f'xi = {par.xi:.4f}, Sigma = {par.Sigma:.4f}')

### 1(b) Household problem

Complete the inversion of the Euler equation (`c_endo` and `m_endo`) in `household_problem.py`.

### 1(c) Wealth distribution

Complete `blocks_ss` and `find_ss_calibrate` in `steady_state.py`.

In [ ]:
model.find_ss(method='calibrate',do_print=True)

In [ ]:
print(f'beta = {par.beta:.4f}, gamma = {par.gamma:.4f}, a_bar = {par.a_bar:.4f}, Gamma = {par.Gamma:.4f}, delta = {par.delta:.4f}')

shares = tails.top_shares(ss.a,ss.D,tops=(0.1,0.01,0.001,0.0001))
pd.DataFrame({'model':[shares['bottom 50%'],shares['top 10%'],shares['top 1%'],shares['top 0.1%'],shares['top 0.01%']],
              'data 1970':[data_1970['Bottom 50%'],data_1970['Top 10%'],data_1970['Top 1%'],data_1970['Top 0.1%'],data_1970['Top 0.01%']]},
             index=['bottom 50% (target)','top 10% (target)','top 1%','top 0.1%','top 0.01%']).round(3)

### 1(d) Policy functions

In [ ]:
# a. analytical approximation for the rich, c = B*a**(Sigma/sigma)
B = np.nan # write your code here
print(f'B = {B:.4f}')

# b. policy functions of alive households, one color per permanent type
alive = np.nonzero(par.newborn_grid == 0)[0]
labels = ['bottom 50%','next 40%','top 10-1%','top 1-0.1%','top 0.1%']
fig,axes = plt.subplots(1,2,figsize=(11,4))
for i_fix in range(par.Nfix):
    for k,i_z in enumerate(alive):
        label = labels[i_fix] if k == 0 else None
        net_savings = ss.a[i_fix,i_z]-par.a_grid
        positive = net_savings > 0
        axes[0].loglog(par.a_grid[1:],ss.c[i_fix,i_z,1:],color=f'C{i_fix}',lw=0.8,label=label)
        axes[1].loglog(par.a_grid[positive],net_savings[positive],color=f'C{i_fix}',lw=0.8)
axes[0].loglog(par.a_grid[1:],B*par.a_grid[1:]**(par.Sigma/par.sigma),color='black',ls='--',label='$Ba_{-}^{\\Sigma/\\sigma}$')
axes[1].loglog(par.a_grid[1:],ss.r*par.a_grid[1:],color='black',ls='--',label='$ra_{-}$')
axes[0].set_title('consumption')
axes[1].set_title('net savings, $a-a_{-}$')
for ax in axes:
    ax.set_xlabel('beginning-of-period wealth, $a_{-}$')
    ax.set_xlim(1e-1,1e8) # savings hit the top of the grid above
axes[0].legend(fontsize=8)
axes[1].legend()
fig.tight_layout()
fig.savefig('figs/policy_functions.pdf',bbox_inches='tight')

In [ ]:
# c. consumption relative to wealth falls with wealth, so the rich save almost all their income
for a_level in [10,100,1_000,10_000,100_000]:
    i = np.searchsorted(par.a_grid,a_level)
    c_model = np.mean(ss.c[:,alive,i])
    print(f'a = {par.a_grid[i]:9.0f}: c/a in the model {100*c_model/par.a_grid[i]:.2f}%, approximation {100*B*par.a_grid[i]**(par.Sigma/par.sigma-1):.2f}%, saving rate out of income {100*(1-c_model/(ss.r*par.a_grid[i]+np.mean(ss.w*(1-par.tau_l)*par.Z_grid))):.1f}%')

### 1(e) Perfect competition

In [ ]:
mu12_1970 = model
mu1_1970 = model.copy()
# write your code here: the economy with mu = 1, with beta re-solved such that r = 3%

pd.DataFrame({label:{'beta':m.par.beta,'gamma':m.par.gamma,'a_bar':m.par.a_bar,'Gamma':m.par.Gamma,'delta':m.par.delta,
                     'r':m.ss.r,'Y':m.ss.Y,'w':m.ss.w,'K/Y':m.ss.K/m.ss.Y,'Q':m.ss.q/m.ss.K,'W/Y':m.ss.A/m.ss.Y,
                     'top 10%':tails.top_shares(m.ss.a,m.ss.D)['top 10%'],'top 1%':tails.top_shares(m.ss.a,m.ss.D)['top 1%'],
                     'top 0.1%':tails.top_shares(m.ss.a,m.ss.D,tops=(0.001,))['top 0.1%']}
              for label,m in [('mu=1.2',mu12_1970),('mu=1',mu1_1970)]}).round(4)

## 2. Steady-state impact of rising permanent income inequality

### 2(a) Stationary equilibrium

Numerical check of Walras' law:

In [ ]:
for label,m in [('mu=1.2',mu12_1970),('mu=1',mu1_1970)]:
    print(f'{label}: asset market {m.ss.clearing_A:.1e}, goods market {m.ss.clearing_Y:.1e}')

### 2(b) Tobin's Q

Numerical check of the formula:

In [ ]:
for label,m in [('mu=1.2',mu12_1970),('mu=1',mu1_1970)]:
    Q_formula = (1-m.par.tau_d)*(1+m.par.alpha*(1-1/m.par.mu)*(m.ss.Y/m.ss.K)/m.ss.r)
    print(f'{label}: Q = {m.ss.q/m.ss.K:.4f}, formula = {Q_formula:.4f}')

### 2(c) Linear approximation

In [ ]:
# partial equilibrium: households get the 2019 grid but face the 1970 prices
def partial_equilibrium(m_1970,Z_grid):
    # write your code here: copy m_1970, give it the new grid, and solve and simulate the households (no market clearing)

    return pe

def elasticities(m,Z_new,h=1e-4):
    """ partial equilibrium change in asset demand and the semi-elasticities of asset demand, asset supply and capital """

    par = m.par
    ss = m.ss

    # a. all blocks at r+h: K from the investment condition, then obj_ss solves firms, mutual fund, government and households
    K_h = np.nan # write your code here: K such that r = ss.r+h, given Gamma, delta and mu
    shifted = m.copy()
    # write your code here: call steady_state.obj_ss at K_h on the copy (the asset market does not have to clear)

    # b. semi-elasticities by finite differences in logs (asset supply is q+B = A, asset demand is A_hh)
    epsilon_K = np.nan # write your code here: -dlog K/dr
    epsilon_q = np.nan # write your code here: -dlog A/dr
    epsilon_A = np.nan # write your code here: dlog A_hh/dr

    # c. partial equilibrium change in asset demand: new grid, old prices
    pe = partial_equilibrium(m,Z_new)
    dlogA = np.nan # write your code here

    return dlogA,epsilon_A,epsilon_q,epsilon_K

linear = {}
for label,m_1970 in [('mu=1.2',mu12_1970),('mu=1',mu1_1970)]:

    dlogA,epsilon_A,epsilon_q,epsilon_K = elasticities(m_1970,Z_2019)
    dr = -dlogA/(epsilon_A+epsilon_q)

    linear[label] = {'dlogA (PE)':dlogA,'epsilon_A':epsilon_A,'epsilon_q':epsilon_q,'epsilon_K':epsilon_K,
                     'dr':dr,'dlog w':-m_1970.par.alpha*epsilon_K*dr,
                     'dlog W/Y':(epsilon_q-m_1970.par.alpha*epsilon_K)/(epsilon_A+epsilon_q)*dlogA}

pd.DataFrame(linear).round(4)

### 2(d) Rising permanent income inequality

#### (i) 2019 steady states

In [ ]:
def solve_economy(base,Z_grid):
    """ steady state of a copy of base with a new permanent income grid """

    # write your code here

    return new

mu12_2019 = solve_economy(mu12_1970,Z_2019)
mu1_2019 = solve_economy(mu1_1970,Z_2019)

#### (ii) Comparison

In [ ]:
def summary(m):
    sh = tails.top_shares(m.ss.a,m.ss.D)
    return {'r':m.ss.r,'w':m.ss.w,'K/Y':m.ss.K/m.ss.Y,'Q':m.ss.q/m.ss.K,'W/Y':m.ss.A/m.ss.Y,
            'bottom 50%':sh['bottom 50%'],'top 10%':sh['top 10%'],'top 1%':sh['top 1%'],'top 0.1%':sh['top 0.1%']}

pd.DataFrame({'mu=1.2, 1970':summary(mu12_1970),'mu=1.2, 2019':summary(mu12_2019),
              'mu=1, 1970':summary(mu1_1970),'mu=1, 2019':summary(mu1_2019)}).round(3)

In [ ]:
# exact changes against the linear approximation of question 2(c)
rows = {}
for label,m_1970,m_2019 in [('mu=1.2',mu12_1970,mu12_2019),('mu=1',mu1_1970,mu1_2019)]:
    rows[label] = {'dr, linear':linear[label]['dr'],'dr, exact':m_2019.ss.r-m_1970.ss.r,
                   'dlog w, linear':linear[label]['dlog w'],'dlog w, exact':np.log(m_2019.ss.w/m_1970.ss.w),
                   'dlog W/Y, linear':linear[label]['dlog W/Y'],'dlog W/Y, exact':np.log((m_2019.ss.A/m_2019.ss.Y)/(m_1970.ss.A/m_1970.ss.Y))}

pd.DataFrame(rows).round(4)

#### (iii) Impact of the change in prices

In [ ]:
# prices only: households have the 1970 grid and face the 2019 prices, markets do not clear
def prices_only(m_2019,Z_grid):
    # write your code here

    return m

def top_wealth_to_output(m,top):
    return tails.average_top_wealth(m.ss.a,m.ss.D,top)/m.ss.Y

rows = {}
for label,m_1970,m_2019 in [('mu=1.2',mu12_1970,mu12_2019),('mu=1',mu1_1970,mu1_2019)]:

    m_prices = prices_only(m_2019,Z_1970)

    for top in [0.01,0.001]:
        base = top_wealth_to_output(m_1970,top)
        rows[(label,f'top {100*top:g}% average wealth/Y (%)')] = {'total':100*(top_wealth_to_output(m_2019,top)/base-1),
                                                                 'prices only':100*(top_wealth_to_output(m_prices,top)/base-1)}

    base_share = tails.top_shares(m_1970.ss.a,m_1970.ss.D)['top 1%']
    rows[(label,'top 1% share (pp)')] = {'total':100*(tails.top_shares(m_2019.ss.a,m_2019.ss.D)['top 1%']-base_share),
                                        'prices only':100*(tails.top_shares(m_prices.ss.a,m_prices.ss.D)['top 1%']-base_share)}

    del m_prices # temporary model

print('changes relative to 1970: total, and with the 1970 grid and the 2019 prices (average wealth relative to output)')
pd.DataFrame(rows).T.round(2)

## 3. One-time shock

### 3(a) Firm value

Complete `production_firm` and the computation of $q_t$ in `mutual_fund` in `blocks.py`.

### 3(b) Transition

In [ ]:
def transition(m_old,m_new):
    """ perfect foresight transition from the steady state of m_old to that of m_new """

    # write your code here

    return m_new.path # no copy: the path stays with m_new, so use a new model for each transition

def path_statistic(path,statistic,T=T_plot):
    """ a statistic of the wealth distribution in each period """

    return np.array([statistic(path.a[t],path.D[t]) for t in range(T)])

def top1_share(a,D):
    return tails.top_shares(a,D)['top 1%']

path_mu12 = transition(mu12_1970,mu12_2019)
path_mu1 = transition(mu1_1970,mu1_2019)

for label,path in [('mu=1.2',path_mu12),('mu=1',path_mu1)]:
    print(f'{label}: return on wealth in the first period = {100*path.ra[0,0]:.2f}%, max |goods market error| = {np.max(np.abs(path.clearing_Y)):.1e}')

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(10,7))
for label,path,m_1970 in [('$\\mu=1.2$',path_mu12,mu12_1970),('$\\mu=1$',path_mu1,mu1_1970)]:
    Q_1970 = m_1970.ss.q/m_1970.ss.K
    WY_1970 = m_1970.ss.A/m_1970.ss.Y
    axes[0,0].plot(100*(path.ra[:T_plot,0]-m_1970.ss.r),label=label)
    axes[0,1].plot(100*(path.q[:T_plot,0]/path.K[:T_plot,0]/Q_1970-1),label=label)
    axes[1,0].plot(100*(path.A[:T_plot,0]/path.Y[:T_plot,0]/WY_1970-1),label=label)
    axes[1,1].plot(100*(path_statistic(path,top1_share)-top1_share(m_1970.ss.a,m_1970.ss.D)),label=label)
axes[0,0].set_title('return on wealth, $r^a_t$ (pp)')
axes[0,1].set_title("average Tobin's Q, $q_t/K_t$ (%)")
axes[1,0].set_title('wealth-output ratio (%)')
axes[1,1].set_title('top 1% wealth share (pp)')
for ax in axes.flat: ax.set_xlabel('years after the shock')
axes[0,0].legend()
fig.tight_layout()
fig.savefig('figs/transition.pdf',bbox_inches='tight')

In [ ]:
top1_1970 = top1_share(mu12_1970.ss.a,mu12_1970.ss.D)
print(f'mu=1.2: firm value on impact {100*(path_mu12.q[0,0]/mu12_1970.ss.q-1):.1f}%, Q from {mu12_1970.ss.q/mu12_1970.ss.K:.2f} to {path_mu12.q[0,0]/path_mu12.K[0,0]:.2f}')
print(f'mu=1.2: capital gain per unit of equity exposure {100*path_mu12.cg[0,0]:.1f}%')
print(f'mu=1.2: wealth-output ratio on impact {100*((path_mu12.A[0,0]/path_mu12.Y[0,0])/(mu12_1970.ss.A/mu12_1970.ss.Y)-1):.1f}%')
print(f'mu=1.2: top 1% share {100*(top1_share(path_mu12.a[0],path_mu12.D[0])-top1_1970):.1f}pp on impact, {100*(top1_share(mu12_2019.ss.a,mu12_2019.ss.D)-top1_1970):.1f}pp in the long run')

### 3(c) General equilibrium effects along the transition (hard)

Changes in the average wealth of the top 1% and 0.1% relative to 1970. Partial equilibrium: 2019 grid and 1970 prices. General equilibrium: 1970 grid and the realized prices.

In [ ]:
prices = ['rB','cg','w','bequest']

def decompose(m_1970,path,top):
    """ change in the average wealth of the top fraction relative to output: total, partial and general equilibrium """

    # a. partial equilibrium: households with the 2019 grid face the 1970 prices (use_inputs=[] keeps all prices at their steady state values)
    m_pe = None # write your code here: a copy of m_1970 with the 2019 grid and its households solved (see partial_equilibrium in question 2(c))
    path_pe = None # write your code here: m_pe.decompose_hh_path(...), starting from the 1970 distribution

    # b. general equilibrium: households with the 1970 grid face the realized prices of the transition
    custom_paths = {varname:path.__dict__[varname][:,0] for varname in prices}
    path_ge = None # write your code here: m_1970.decompose_hh_path(...) with these prices as inputs

    # c. average wealth relative to output (realized output, or 1970 output when prices are fixed), relative to 1970
    def average(a,D): return tails.average_top_wealth(a,D,top)
    base = average(m_1970.ss.a,m_1970.ss.D)/m_1970.ss.Y
    Y = path.Y[:T_plot,0]
    return {'total':path_statistic(path,average)/Y/base-1,
            'partial equilibrium':path_statistic(path_pe,average)/m_1970.ss.Y/base-1,
            'general equilibrium':path_statistic(path_ge,average)/Y/base-1}

decomposition = {(mu,top):decompose(m_1970,path,top) for mu,m_1970,path in [(1.2,mu12_1970,path_mu12),(1.0,mu1_1970,path_mu1)] for top in [0.01,0.001]}

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,mu in [(axes[0],1.2),(axes[1],1.0)]:
    for key,value in decomposition[(mu,0.001)].items():
        ax.plot(100*value,label=key)
    ax.axhline(0.0,color='black',lw=0.5)
    ax.set_title(f'average wealth/Y of the top 0.1% (%), $\\mu={mu}$')
    ax.set_xlabel('years after the shock')
axes[0].legend()
fig.tight_layout()
fig.savefig('figs/decomposition.pdf',bbox_inches='tight')

fig,axes = plt.subplots(1,2,figsize=(11,4))
for ax,top in [(axes[0],0.01),(axes[1],0.001)]:
    for mu in [1.2,1.0]:
        ax.plot(100*decomposition[(mu,top)]['general equilibrium'],label=f'$\\mu={mu}$')
    ax.axhline(0.0,color='black',lw=0.5)
    ax.set_title(f'general equilibrium effect, top {100*top:g}% (%)')
    ax.set_xlabel('years after the shock')
axes[0].legend()
fig.tight_layout()
fig.savefig('figs/general_equilibrium.pdf',bbox_inches='tight')

for mu in [1.2,1.0]:
    ge = decomposition[(mu,0.001)]['general equilibrium']
    sign_change = np.argmax(ge < 0) if np.any(ge < 0) else None
    print(f'mu = {mu}: general equilibrium effect on the top 0.1% on impact {100*ge[0]:.1f}%, after 20 years {100*ge[20]:.1f}%, after 100 years {100*ge[99]:.1f}%, negative from year {sign_change}')

## 4. Myopic expectations (optional)

Every 5 years the permanent income grid moves one step along a linear path from 1970 to 2019, and households and firms believe each step is permanent. Each step is an unexpected shock: solve the steady state that households believe in, and the transition to it from the current state. Store the 10 transitions in the list `trans_myopic`, keeping only the aggregates and the first `T_plot` periods of `a` and `D`; the given code below joins them.

In [ ]:
benchmark = mu12_1970
trans_myopic = []

ini_myopic = {} # write your code here: the initial state in 1970

Z_path = np.linspace(Z_1970,Z_2019,11)

for t in range(1,11):

    # a. steady state households believe in
    # write your code here

    # b. transition from the current state (the model is called _model here)
    # write your code here

    # only the aggregates and the first T_plot periods of a and D are kept: a full path takes 0.35 GB
    trans_myopic.append({varname:getattr(_model.path,varname)[:,0].copy() for varname in ['ra','q','K','A','Y','r','w']})
    trans_myopic[-1]['a'] = _model.path.a[:T_plot].copy()
    trans_myopic[-1]['D'] = _model.path.D[:T_plot].copy()

    # c. the state after 5 years is the starting point of the next shock
    ini_myopic = {} # write your code here

del _model # temporary model

In [ ]:
# given code: the realized path from the 10 transitions in trans_myopic, each lasting 5 years and the last one until the end
T = benchmark.par.T

full_trans = {}
for varname in ['ra','q','K','A','Y','r','w']:
    full_trans[varname] = trans_myopic[0][varname].copy()
    for i in range(1,10):
        full_trans[varname][5*i:] = trans_myopic[i][varname][:T-5*i]

full_trans['Q'] = full_trans['q']/full_trans['K']

# average wealth of the top 0.1%, from the transition running in each year
full_trans['top01'] = np.zeros(T_plot)
for t in range(T_plot):
    i = min(t//5,9)
    full_trans['top01'][t] = tails.average_top_wealth(trans_myopic[i]['a'][t-5*i],trans_myopic[i]['D'][t-5*i],0.001)

In [ ]:
def top01_wealth(a,D):
    return tails.average_top_wealth(a,D,0.001)

def get_pc_myopic(trans,var,ss_value):
    return 100*(trans[var]-ss_value)/ss_value

def get_ac_myopic(trans,var,ss_value):
    return 100*(trans[var]-ss_value)

one_time = {'ra':path_mu12.ra[:,0],'Q':path_mu12.q[:,0]/path_mu12.K[:,0],'top01':path_statistic(path_mu12,top01_wealth)}
Q_ss = benchmark.ss.q/benchmark.ss.K
top01_ss = top01_wealth(benchmark.ss.a,benchmark.ss.D)

fig,axs = plt.subplots(1,3,figsize=(13,4))
for trans,label,style in [(one_time,'one-time','-'),(full_trans,'myopic',':')]:
    axs[0].plot(get_ac_myopic(trans,'ra',benchmark.ss.r)[:T_plot],ls=style,label=label)
    axs[1].plot(get_pc_myopic(trans,'Q',Q_ss)[:T_plot],ls=style,label=label)
    axs[2].plot(get_pc_myopic(trans,'top01',top01_ss)[:T_plot],ls=style,label=label)
axs[0].set_title('return on wealth, $r^a_t$ (pp)')
axs[1].set_title("average Tobin's Q (%)")
axs[2].set_title('average wealth of the top 0.1% (%)')
for ax in axs: ax.set_xlabel('years after 1970')
axs[0].legend()
fig.tight_layout()
fig.savefig('figs/myopic.pdf',bbox_inches='tight')

for t in [10,30,50]:
    print(f'year {t}: top 0.1% average wealth, one-time {get_pc_myopic(one_time,"top01",top01_ss)[t]:.1f}%, myopic {get_pc_myopic(full_trans,"top01",top01_ss)[t]:.1f}%')